# Sentinel-CCTV — Traffic Event Detection & Accident Anticipation
### WIUT Hackathon 2026 — Computer Vision Track
**Team:** DeepFlow-Vision (Shermuxamedov Bekzod, Voronin Mixail, Xadjiaxmedova Hilolabonu)

This notebook provides end-to-end exploratory data analysis, visual geometry calibration, trajectory analysis, event detection, causal risk anticipation, and official evaluation metric validation.

In [ ]:
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import solution
from src.scene import SceneGeometry, TrafficLightTracker
from src.tracker import RoadTracker, Detection
from src.event_engine import EventEngine
from src.risk_estimator import RiskEstimator
import evaluate

## 1. Video Inspection & Scene Geometry
Inspect fixed CCTV video properties and overlay the calibrated intersection geometry.

In [ ]:
video_path = "samples/sample_004.mp4"
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 29.97
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = n_frames / fps
print(f"Resolution: {w}x{h} @ {fps:.2f} FPS | Total Frames: {n_frames} | Duration: {duration:.2f}s")
ret, frame = cap.read()
cap.release()

scene = SceneGeometry(w, h)
overlay = frame.copy()
cv2.line(overlay, tuple(scene.stop_line[0].astype(int)), tuple(scene.stop_line[1].astype(int)), (0, 255, 255), 6)
cv2.fillPoly(overlay, [scene.crosswalk_main.astype(int)], (0, 255, 0))
cv2.fillPoly(overlay, [scene.crosswalk_right.astype(int)], (255, 255, 0))
cv2.fillPoly(overlay, [scene.crosswalk_bottom.astype(int)], (255, 0, 255))
cv2.fillPoly(overlay, [scene.island_refuge.astype(int)], (0, 165, 255))
vis = cv2.addWeighted(overlay, 0.35, frame, 0.65, 0)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Calibrated Scene Geometry: Stop Line, Crosswalks and Refuge Island")
plt.axis("off")
plt.show()

## 2. Part A: Event Detection
Run `solution.detect_events` to extract official traffic events as tight temporal segments.

In [ ]:
events = solution.detect_events(video_path)
print(f"Detected {len(events)} events:")
for s, e, lab in events:
    print(f"  [{s:6.2f}s - {e:6.2f}s]: {lab}")

## 3. Official Validation & Scoring against Dev Ground Truth
Validate prediction schema and compute macro F1 across tIoU thresholds (0.3, 0.5, 0.7).

In [ ]:
pred_file = "predictions_samples.json"
gt_file = "dev_ground_truth.json"
with open(pred_file) as f:
    pred = json.load(f)
with open(gt_file) as f:
    gt = json.load(f)
errs, warns = evaluate.validate(pred, gt)
print(f"Format check: {len(errs)} errors, {len(warns)} warnings")
report = evaluate.evaluate_part_a(gt, pred["videos"], per_video=True)
print(f"Part A Model Score: {report['score_a']:.4f}")
for c, metrics in report["per_class"].items():
    print(f"  {c:20s}: F1@0.3={metrics['0.3']['f1']:.3f}, F1@0.5={metrics['0.5']['f1']:.3f}, F1@0.7={metrics['0.7']['f1']:.3f}, mean={metrics['f1_mean']:.3f}")